# SQL Server to Unity Catalog Volume Ingestion

Reads tables from SQL Server via JDBC and writes them as Parquet files to a Unity Catalog Volume.

In [0]:
# ── SQL Server connection settings ──────────────────────────────────────────
# Store credentials in Databricks Secrets and reference them below.
# To create a secret: databricks secrets put-secret <scope> <key>

SQL_HOST     = "managed-connector-demo-server.database.windows.net"      # e.g. myserver.database.windows.net
SQL_PORT     = 1433
SQL_DATABASE = "managed-connector-demo-db"
SQL_USER     = "sqladmin"
SQL_PASSWORD = "xxxxx!"

# Tables to ingest (list of "schema.table" strings)
SOURCE_TABLES = [
    "dbo.customers",
    "dbo.orders",
]

# ── Destination Unity Catalog Volume ─────────────────────────────────────────
VOLUME_PATH   = "/Volumes/ext_cat/sql_db/staging"   # base path
OUTPUT_FORMAT = "delta"   # parquet | delta | csv | json

In [0]:
from pyspark.sql import SparkSession

jdbc_url = (
    f"jdbc:sqlserver://{SQL_HOST}:{SQL_PORT};"
    f"databaseName={SQL_DATABASE};"
    "encrypt=true;trustServerCertificate=false;"
    "loginTimeout=30;"
)

connection_properties = {
    "user":     SQL_USER,
    "password": SQL_PASSWORD,
    "driver":   "com.microsoft.sqlserver.jdbc.SQLServerDriver",
}

# Quick connectivity check — list tables in the target database
test_df = spark.read.jdbc(
    url=jdbc_url,
    table="(SELECT TOP 1 1 AS ok) AS ping",
    properties=connection_properties,
)
test_df.show()
print("✅ Connection to SQL Server successful")

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8345338825761548>, line 22
     16 # Quick connectivity check — list tables in the target database
     17 test_df = spark.read.jdbc(
     18     url=jdbc_url,
     19     table="(SELECT TOP 1 1 AS ok) AS ping",
     20     properties=connection_properties,
     21 )
---> 22 test_df.show()
     23 print("✅ Connection to SQL Server successful")

File <command-8345338825761548>, line 17
     10 connection_properties = {
     11     "user":     SQL_USER,
     12     "password": SQL_PASSWORD,
     13     "driver":   "com.microsoft.sqlserver.jdbc.SQLServerDriver",
     14 }
     16 # Quick connectivity check — list tables in the target database
---> 17 test_df = spark.read.jdbc(
     18     url=jdbc_url,
     19     table="(SELECT TOP 1 1 AS ok) AS ping",
     20     properties=connection_properties,
     21 )
     22 test_df.s

In [0]:
import os

for source_table in SOURCE_TABLES:
    print(f"Ingesting {source_table} ...")

    # Read full table from SQL Server
    df = spark.read.jdbc(
        url=jdbc_url,
        table=source_table,
        properties=connection_properties,
    )

    row_count = df.count()
    print(f"  Rows read: {row_count}")

    # Derive a safe folder name from "schema.table" -> "schema_table"
    folder_name = source_table.replace(".", "_")
    dest_path = f"{VOLUME_PATH}/{folder_name}"

    # Write to Volume
    writer = df.write.mode("overwrite")

    if OUTPUT_FORMAT == "parquet":
        writer.parquet(dest_path)
    elif OUTPUT_FORMAT == "delta":
        writer.format("delta").save(dest_path)
    elif OUTPUT_FORMAT == "csv":
        writer.option("header", True).csv(dest_path)
    elif OUTPUT_FORMAT == "json":
        writer.json(dest_path)
    else:
        raise ValueError(f"Unsupported OUTPUT_FORMAT: {OUTPUT_FORMAT}")

    print(f"  Written to {dest_path}")
    print()

print("✅ All tables ingested.")

Ingesting dbo.customers ...
  Rows read: 10
  Written to /Volumes/ext_cat/sql_db/staging/dbo_customers

Ingesting dbo.orders ...
  Rows read: 10
  Written to /Volumes/ext_cat/sql_db/staging/dbo_orders

✅ All tables ingested.


In [0]:
# Verify the files were written to the volume
for source_table in SOURCE_TABLES:
    folder_name = source_table.replace(".", "_")
    dest_path = f"{VOLUME_PATH}/{folder_name}"

    files = dbutils.fs.ls(dest_path)
    print(f"{dest_path}: {len(files)} file(s)")
    for f in files[:5]:
        print(f"  {f.name}  ({f.size:,} bytes)")

/Volumes/ext_cat/sql_db/staging/dbo_customers: 2 file(s)
  _delta_log/  (0 bytes)
  part-00000-cc789a97-6df0-420f-a023-e33cc31e772d.c000.snappy.parquet  (4,779 bytes)
/Volumes/ext_cat/sql_db/staging/dbo_orders: 2 file(s)
  _delta_log/  (0 bytes)
  part-00000-bab001c9-b3d5-44a5-bdb9-2145e1652ec3.c000.snappy.parquet  (2,217 bytes)


In [0]:
%sql
select * from delta.`/Volumes/ext_cat/sql_db/staging/dbo_customers/`

customer_id,first_name,last_name,phone,email,address,city,state,zip
101,Amit,Sharma,9876543210,amit.sharma@email.com,12 MG Road,Bengaluru,Karnataka,560001
102,Priya,Nair,9876543211,priya.nair@email.com,45 Kowdiar Road,Thiruvananthapuram,Kerala,695003
103,Rahul,Verma,9876543212,rahul.verma@email.com,78 Park Street,Kolkata,West Bengal,700016
104,Sneha,Iyer,9876543213,sneha.iyer@email.com,23 Anna Nagar,Chennai,Tamil Nadu,600040
105,Arjun,Mehta,9876543214,arjun.mehta@email.com,56 Satellite Road,Ahmedabad,Gujarat,380015
106,Neha,Kapoor,9876543215,neha.kapoor@email.com,89 Sector 17,Chandigarh,Chandigarh,160017
107,Vikram,Reddy,9876543216,vikram.reddy@email.com,34 Banjara Hills,Hyderabad,Telangana,500034
108,Anjali,Patel,9876543217,anjali.patel@email.com,67 FC Road,Pune,Maharashtra,411004
109,Karan,Singh,9876543218,karan.singh@email.com,90 Civil Lines,Jaipur,Rajasthan,302006
110,Meera,Thomas,9876543219,meera.thomas@email.com,15 Marine Drive,Kochi,Kerala,682011


In [0]:
%sql
select * from delta.`/Volumes/ext_cat/sql_db/staging/dbo_orders/`

order_id,order_date,customer_id,order_status
10001,2026-08-01T10:15:00.000Z,101,Delivered
10002,2026-08-03T14:30:00.000Z,102,Shipped
10003,2026-08-05T09:45:00.000Z,103,Delivered
10004,2026-08-07T16:20:00.000Z,104,Processing
10005,2026-08-10T11:10:00.000Z,105,Delivered
10006,2026-08-12T13:50:00.000Z,101,Delivered
10007,2026-08-15T18:05:00.000Z,106,Cancelled
10008,2026-08-18T10:40:00.000Z,107,Shipped
10009,2026-08-20T15:25:00.000Z,104,Processing
10010,2026-08-22T12:00:00.000Z,108,Delivered
